# 09 · Interfaz Gradio

> **Prototipo académico; no apto para diagnóstico ni decisiones clínicas.**

Carga el modelo final (`modelo_final.json`) y el umbral elegido con validación
(`umbral_final.json`) y abre la interfaz. El código de la interfaz está en `src/app_gradio.py`.

- **Formatos admitidos:** PNG, JPG o DICOM (.dcm) del recorte de una lesión en escala de grises.
- **Se rechazan con un mensaje:** otros formatos, archivos dañados, imágenes constantes,
  imágenes de menos de 32 px por lado, imágenes de más de 2000 px por lado (mamografías
  completas o máscaras) y máscaras binarias.
- Los ejemplos son PNG de **validación**, no de prueba.

In [ ]:
!pip install -q gradio pydicom
!pip show gradio | head -2

In [ ]:
# Montar Drive y fijar rutas (las mismas en todos los notebooks)
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE    = "/content/drive/MyDrive/Aprendizaje final"
DATA    = f"{BASE}/Csv"
RES     = f"{BASE}/resultados"
SRC     = f"{BASE}/src"
PART    = f"{RES}/particiones"
PNG     = f"{BASE}/procesadas_224"     # recortes preprocesados (no van a Git)
MODELOS = f"{BASE}/modelos"            # checkpoints .pt (no van a Git)
for p in [DATA, RES, SRC, PART]:
    assert os.path.exists(p), f"No existe: {p}"
sys.path.insert(0, SRC)

In [ ]:
import json, importlib
import numpy as np, pandas as pd, torch
from PIL import Image
import app_gradio as ag
importlib.reload(ag)

final = json.load(open(f"{RES}/modelo_final.json"))
umb = json.load(open(f"{RES}/umbral_final.json"))
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

val = pd.read_csv(f"{PART}/val.csv")
ejemplos = [[f"{PNG}/{val[val.label == c].lesion_id.iloc[0]}.png"] for c in (0, 1)]
app = ag.crear_app(f"{MODELOS}/{final['ckpt']}", umb["umbral"], dispositivo, ejemplos)
print("Modelo:", final["ckpt"], "| umbral:", umb["umbral"])

## Prueba rápida de entradas inválidas (evidencia para el informe)

In [ ]:
import tempfile
tmp = tempfile.mkdtemp()
casos = {
    "texto.txt": None,
    "no_es_imagen.png": None,
    "constante.png": np.full((100, 100), 128, np.uint8),
    "diminuta.png": np.random.randint(0, 255, (10, 10), np.uint8),
    "mascara.png": (np.random.rand(200, 200) > .5).astype(np.uint8) * 255,
    "enorme.png": np.random.randint(0, 255, (3000, 2500), np.uint8),
}
for nombre, arr in casos.items():
    ruta = f"{tmp}/{nombre}"
    if arr is None:
        open(ruta, "w").write("esto no es una imagen")
    else:
        Image.fromarray(arr).save(ruta)
    imagen, texto = app.responder(ruta)
    print(f"{nombre:18s} -> {texto.splitlines()[-1]}")

imagen, texto = app.responder(ejemplos[1][0])
print("\nEjemplo válido:\n", texto)

## La interfaz da la misma puntuación que el notebook
Se comparan 5 lesiones de validación: puntuación de la app frente a `pred_val_final.csv` (notebook 06).

In [ ]:
predecir = ag.crear_predictor(f"{MODELOS}/{final['ckpt']}", dispositivo)
pv = pd.read_csv(f"{RES}/pred_val_final.csv").head(5)
pv["score_app"] = [predecir(f"{PNG}/{l}.png")[1] for l in pv.lesion_id]
pv["diferencia"] = (pv.score_app - pv.score).abs()
print(pv.round(4).to_string(index=False))
print("Diferencia máxima:", round(float(pv.diferencia.max()), 4))

## Abrir la interfaz dentro del notebook
- La interfaz se muestra **debajo de la celda**, a través del proxy de Colab.
- No se usa el enlace público `gradio.live`: en las pruebas del 21/11 daba 404 o 504.
- `ssr_mode=False`: Gradio 6 renderiza en el servidor con un proceso aparte y eso falla detrás
  del proxy de Colab; así la página se arma en el navegador.
- `gr.close_all()` cierra lanzamientos anteriores, que dejaban el puerto ocupado.
- Si aparece `Address already in use`: Entorno de ejecución → Reiniciar sesión y ejecutar desde arriba.

In [ ]:
import gradio as gr, requests, inspect
from google.colab import output

gr.close_all()      # cierra los lanzamientos anteriores
app = ag.crear_app(f"{MODELOS}/{final['ckpt']}", umb["umbral"], dispositivo, ejemplos)

PUERTO = 7870
extra = {"ssr_mode": False} if "ssr_mode" in inspect.signature(app.launch).parameters else {}
app.launch(share=False, inline=False, prevent_thread_lock=True,
           server_name="0.0.0.0", server_port=PUERTO, quiet=True, **extra)

print("Servidor local responde:", requests.get(f"http://127.0.0.1:{PUERTO}").status_code)
output.serve_kernel_port_as_iframe(PUERTO, height=900)

In [ ]:
# Para cerrar la interfaz al terminar
app.close()